# Notebook Goal

This notebook builds and tests a reusable inference pipeline for the Credit Card Fraud Detection project.

Notebook `17_final_model_training.ipynb` already trained and saved the final validated fraud model and its supporting artifacts. Notebook `18_inference_pipeline.ipynb` will load those saved artifacts and use them to make predictions.

No model training happens in this notebook. No threshold tuning happens in this notebook. The selected features and saved decision policy from notebook 17 are reused as-is.

The purpose of this notebook is to simulate how the fraud model will work in production: load the saved model, prepare input data in the expected format, generate a fraud probability, and convert that result into a reusable prediction workflow.

This notebook also prepares the project for the next FastAPI `/predict` endpoint by turning the saved model artifacts into a clear, testable inference path.


# Load Saved Artifacts

This section loads the final validated model and the supporting JSON artifacts created in notebook `17_final_model_training.ipynb`.

The goal here is only to verify that the saved inference assets exist and can be loaded correctly before later steps use them for prediction.


In [69]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier


In [70]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

FINAL_MODEL_PATH = ARTIFACTS_DIR / "final_validated_fraud_model.joblib"
FINAL_FEATURE_COLUMNS_PATH = ARTIFACTS_DIR / "final_feature_columns.json"
FINAL_DECISION_POLICY_PATH = ARTIFACTS_DIR / "final_decision_policy.json"
FINAL_MODEL_METADATA_PATH = ARTIFACTS_DIR / "final_model_metadata.json"
FINAL_MODEL_METRICS_PATH = ARTIFACTS_DIR / "final_model_metrics.json"

artifact_paths = {
    "final_validated_model": FINAL_MODEL_PATH,
    "final_feature_columns": FINAL_FEATURE_COLUMNS_PATH,
    "final_decision_policy": FINAL_DECISION_POLICY_PATH,
    "final_model_metadata": FINAL_MODEL_METADATA_PATH,
    "final_model_metrics": FINAL_MODEL_METRICS_PATH,
}

for artifact_name, artifact_path in artifact_paths.items():
    if not artifact_path.exists():
        raise FileNotFoundError(f"Missing required artifact: {artifact_path}")

    print(f"Found {artifact_name}: {artifact_path}")


Found final_validated_model: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_validated_fraud_model.joblib
Found final_feature_columns: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_feature_columns.json
Found final_decision_policy: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_decision_policy.json
Found final_model_metadata: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_model_metadata.json
Found final_model_metrics: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_model_metrics.json


In [71]:
final_model = joblib.load(FINAL_MODEL_PATH)
print(f"Loaded final validated model successfully: {FINAL_MODEL_PATH}")

with open(FINAL_FEATURE_COLUMNS_PATH, "r", encoding="utf-8") as f:
    final_feature_columns = json.load(f)
print(f"Loaded final feature columns successfully: {FINAL_FEATURE_COLUMNS_PATH}")

with open(FINAL_DECISION_POLICY_PATH, "r", encoding="utf-8") as f:
    final_decision_policy = json.load(f)
print(f"Loaded final decision policy successfully: {FINAL_DECISION_POLICY_PATH}")

with open(FINAL_MODEL_METADATA_PATH, "r", encoding="utf-8") as f:
    final_model_metadata = json.load(f)
print(f"Loaded final model metadata successfully: {FINAL_MODEL_METADATA_PATH}")

with open(FINAL_MODEL_METRICS_PATH, "r", encoding="utf-8") as f:
    final_model_metrics = json.load(f)
print(f"Loaded final model metrics successfully: {FINAL_MODEL_METRICS_PATH}")


Loaded final validated model successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_validated_fraud_model.joblib
Loaded final feature columns successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_feature_columns.json
Loaded final decision policy successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_decision_policy.json
Loaded final model metadata successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_model_metadata.json
Loaded final model metrics successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_model_metrics.json


# Validate Artifact Consistency

This section checks that the loaded model and saved artifacts are aligned before they are used for inference.

If anything important is missing or mismatched, the notebook stops early with a clear validation error.


In [72]:
if not isinstance(final_feature_columns, list) or not final_feature_columns:
    raise ValueError("Artifact validation failed: final_feature_columns must be a non-empty list.")

if not isinstance(final_model, RandomForestClassifier):
    raise ValueError(
        "Artifact validation failed: loaded model must be a RandomForestClassifier. "
        f"Found {type(final_model).__name__}."
    )

if "review_threshold" not in final_decision_policy:
    raise ValueError(
        "Artifact validation failed: final_decision_policy is missing review_threshold."
    )

if "block_threshold" not in final_decision_policy:
    raise ValueError(
        "Artifact validation failed: final_decision_policy is missing block_threshold."
    )

review_threshold = final_decision_policy["review_threshold"]
block_threshold = final_decision_policy["block_threshold"]

if review_threshold >= block_threshold:
    raise ValueError(
        "Artifact validation failed: review_threshold must be less than block_threshold. "
        f"Found review_threshold={review_threshold} and block_threshold={block_threshold}."
    )

actual_feature_count = len(final_feature_columns)

if hasattr(final_model, "n_features_in_"):
    expected_feature_count = final_model.n_features_in_
elif "feature_count" in final_model_metadata:
    expected_feature_count = final_model_metadata["feature_count"]
else:
    raise ValueError(
        "Artifact validation failed: could not determine the model's expected feature count."
    )

if expected_feature_count != actual_feature_count:
    raise ValueError(
        "Artifact validation failed: model feature count does not match final_feature_columns. "
        f"Expected {expected_feature_count}, found {actual_feature_count}."
    )

model_version = final_model_metadata.get("model_version") or final_model_metadata.get("version")

print(f"Model type: {type(final_model).__name__}")
print(f"Expected feature count: {expected_feature_count}")
print(f"Actual feature count: {actual_feature_count}")
print(f"Review threshold: {review_threshold}")
print(f"Block threshold: {block_threshold}")
print(f"Model version: {model_version if model_version is not None else 'Not available'}")


Model type: RandomForestClassifier
Expected feature count: 13
Actual feature count: 13
Review threshold: 0.35
Block threshold: 0.5
Model version: Not available


# Create Input Validation Function

This section defines a small helper that validates one transaction before any inference step uses it.

The function checks required features, verifies numeric values, and safely ignores extra input columns.


In [73]:
def validate_transaction_input(transaction, feature_columns):
    if isinstance(transaction, pd.Series):
        transaction_data = transaction.to_dict()
    elif isinstance(transaction, dict):
        transaction_data = transaction
    else:
        raise ValueError(
            "Transaction input validation failed: transaction must be a dict or pandas Series."
        )

    if not transaction_data:
        raise ValueError(
            "Transaction input validation failed: transaction input cannot be empty."
        )

    missing_features = [
        feature_name for feature_name in feature_columns if feature_name not in transaction_data
    ]
    if missing_features:
        raise ValueError(
            "Transaction input validation failed: missing required features: "
            f"{missing_features}"
        )

    non_numeric_features = []
    for feature_name in feature_columns:
        feature_value = transaction_data[feature_name]

        if isinstance(feature_value, bool) or not isinstance(
            feature_value, (int, float, np.integer, np.floating)
        ):
            non_numeric_features.append(
                f"{feature_name}={feature_value!r} ({type(feature_value).__name__})"
            )

    if non_numeric_features:
        raise ValueError(
            "Transaction input validation failed: required features must be numeric. "
            f"Found invalid values: {non_numeric_features}"
        )

    extra_columns = [
        column_name for column_name in transaction_data if column_name not in feature_columns
    ]
    if extra_columns:
        print(f"Warning: ignoring extra input columns: {extra_columns}")

    return True


# Create Preprocessing Function

This section prepares one validated transaction in the exact feature order expected by the saved model.

The helper always follows the saved `final_feature_columns.json` order so later prediction steps never depend on random input ordering.


In [74]:
def prepare_model_input(transaction, feature_columns):
    validate_transaction_input(transaction, feature_columns)

    if isinstance(transaction, pd.Series):
        transaction_data = transaction.to_dict()
    else:
        transaction_data = dict(transaction)

    # Filter to required features only; reindex below enforces saved artifact column order.
    filtered_transaction = {
        key: value for key, value in transaction_data.items() if key in feature_columns
    }

    model_input_df = pd.DataFrame([filtered_transaction])
    model_input_df = model_input_df.reindex(columns=feature_columns)
    model_input_df = model_input_df.apply(pd.to_numeric, errors="raise")

    if model_input_df.columns.tolist() != feature_columns:
        raise ValueError(
            "Model input preparation failed: DataFrame columns do not match the saved feature order."
        )

    expected_shape = (1, len(feature_columns))
    if model_input_df.shape != expected_shape:
        raise ValueError(
            "Model input preparation failed: prepared DataFrame has the wrong shape. "
            f"Expected {expected_shape}, found {model_input_df.shape}."
        )

    if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in model_input_df.dtypes):
        raise ValueError(
            "Model input preparation failed: prepared DataFrame contains non-numeric values."
        )

    return model_input_df


In [75]:
processed_sample_path = PROJECT_ROOT / "data" / "processed" / "final_features.csv"

if processed_sample_path.exists():
    sample_source_df = pd.read_csv(processed_sample_path, nrows=1)
    sample_transaction = sample_source_df.drop(columns=["Class"], errors="ignore").iloc[0].to_dict()
    sample_source = f"processed sample from {processed_sample_path.name}"
else:
    sample_transaction = {
        feature_name: float(index)
        for index, feature_name in enumerate(final_feature_columns, start=1)
    }
    sample_source = "dummy sample built from final_feature_columns"

sample_transaction["extra_input_column"] = 999.0

prepared_sample_input = prepare_model_input(sample_transaction, final_feature_columns)

assert prepared_sample_input.shape == (1, len(final_feature_columns))
assert prepared_sample_input.columns.tolist() == final_feature_columns
assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in prepared_sample_input.dtypes)

print(f"Prepared sample source: {sample_source}")
print(prepared_sample_input.shape)
print(prepared_sample_input.columns.tolist())
prepared_sample_input.head()


Prepared sample source: processed sample from final_features.csv
(1, 13)
['V14_V12_interaction', 'V14', 'V17_V16_interaction', 'V12', 'V17', 'V10', 'V4', 'V16', 'V3', 'V11', 'V7', 'V18', 'log_amount']


,V14_V12_interaction,V14,V17_V16_interaction,V12,V17,V10,V4,V16,V3,V11,V7,V18,log_amount
0,0.192241,-0.311169,-0.09783,-0.617801,0.207971,0.090794,1.378155,-0.470401,2.536347,-0.5516,0.239599,0.025791,5.01476


# Create Prediction Function

This section applies the saved business thresholds and adds a small prediction helper that returns an API-ready response.

`apply_decision_policy` uses the thresholds loaded from `final_decision_policy.json`, and `predict_fraud` reuses that policy after scoring one transaction.


In [76]:
def apply_decision_policy(probability, policy):
    review_threshold = policy["review_threshold"]
    block_threshold = policy["block_threshold"]

    if probability >= block_threshold:
        return {
            "decision": "BLOCK",
            "risk_level": "HIGH",
            "reason": "Transaction probability is above block threshold.",
        }

    if probability >= review_threshold:
        return {
            "decision": "REVIEW",
            "risk_level": "MEDIUM",
            "reason": "Transaction probability is between review and block thresholds.",
        }

    return {
        "decision": "APPROVE",
        "risk_level": "LOW",
        "reason": "Transaction probability is below review threshold.",
    }


def predict_fraud(transaction):
    input_df = prepare_model_input(transaction, final_feature_columns)
    fraud_probability = float(final_model.predict_proba(input_df)[0][1])

    decision_result = apply_decision_policy(fraud_probability, final_decision_policy)
    model_version = final_model_metadata.get("model_version") or final_model_metadata.get("version")

    return {
        "fraud_probability": fraud_probability,
        "decision": decision_result["decision"],
        "risk_level": decision_result["risk_level"],
        "reason": decision_result["reason"],
        "thresholds": {
            "review_threshold": final_decision_policy["review_threshold"],
            "block_threshold": final_decision_policy["block_threshold"],
        },
        "model_version": model_version if model_version is not None else "Not available",
    }


In [77]:
sample_prediction = predict_fraud(sample_transaction)
sample_prediction


{'fraud_probability': 0.0,
 'decision': 'APPROVE',
 'risk_level': 'LOW',
 'reason': 'Transaction probability is below review threshold.',
 'thresholds': {'review_threshold': 0.35, 'block_threshold': 0.5},
 'model_version': 'Not available'}

In [78]:
block_test   = final_decision_policy["block_threshold"]
review_test  = final_decision_policy["review_threshold"]
approve_test = round(final_decision_policy["review_threshold"] - 0.01, 4)

block_result   = apply_decision_policy(block_test,   final_decision_policy)
review_result  = apply_decision_policy(review_test,  final_decision_policy)
approve_result = apply_decision_policy(approve_test, final_decision_policy)

assert block_result["decision"]   == "BLOCK"  and block_result["risk_level"]   == "HIGH"
assert review_result["decision"]  == "REVIEW" and review_result["risk_level"]  == "MEDIUM"
assert approve_result["decision"] == "APPROVE" and approve_result["risk_level"] == "LOW"

print(f"BLOCK   (p={block_test}):   {block_result}")
print(f"REVIEW  (p={review_test}):  {review_result}")
print(f"APPROVE (p={approve_test}): {approve_result}")


BLOCK   (p=0.5):   {'decision': 'BLOCK', 'risk_level': 'HIGH', 'reason': 'Transaction probability is above block threshold.'}
REVIEW  (p=0.35):  {'decision': 'REVIEW', 'risk_level': 'MEDIUM', 'reason': 'Transaction probability is between review and block thresholds.'}
APPROVE (p=0.34): {'decision': 'APPROVE', 'risk_level': 'LOW', 'reason': 'Transaction probability is below review threshold.'}


# Test with Sample Transactions

This section uses a few sample transactions only to prove that the saved inference pipeline works end to end.

It does not retrain the model, tune thresholds, or recalculate overall performance metrics.

In [79]:
phase8_dataset_path = PROJECT_ROOT / "data" / "processed" / "final_features.csv"
phase8_samples = []

if phase8_dataset_path.exists():
    phase8_df = pd.read_csv(phase8_dataset_path)

    normal_rows = phase8_df[phase8_df["Class"] == 0]
    fraud_rows = phase8_df[phase8_df["Class"] == 1]

    if not normal_rows.empty:
        normal_transaction = normal_rows.iloc[0][final_feature_columns].to_dict()
        phase8_samples.append(
            {
                "sample_type": "Actual normal transaction",
                "actual_class": int(normal_rows.iloc[0]["Class"]),
                "transaction": normal_transaction,
            }
        )

    if not fraud_rows.empty:
        fraud_transaction = fraud_rows.iloc[0][final_feature_columns].to_dict()
        phase8_samples.append(
            {
                "sample_type": "Actual fraud transaction",
                "actual_class": int(fraud_rows.iloc[0]["Class"]),
                "transaction": fraud_transaction,
            }
        )

    if not normal_rows.empty and not fraud_rows.empty:
        normal_series = normal_rows.iloc[0][final_feature_columns]
        fraud_series = fraud_rows.iloc[0][final_feature_columns]
        borderline_transaction = ((normal_series + fraud_series) / 2.0).to_dict()
        phase8_samples.append(
            {
                "sample_type": "Manually created borderline transaction",
                "actual_class": "Not available",
                "transaction": borderline_transaction,
            }
        )

if not phase8_samples:
    dummy_base = {
        feature_name: float(index)
        for index, feature_name in enumerate(final_feature_columns, start=1)
    }

    phase8_samples = [
        {
            "sample_type": "Dummy smoke test 1",
            "actual_class": "Not available",
            "transaction": dummy_base,
        },
        {
            "sample_type": "Dummy smoke test 2",
            "actual_class": "Not available",
            "transaction": {
                feature_name: value + 0.5 for feature_name, value in dummy_base.items()
            },
        },
        {
            "sample_type": "Dummy smoke test 3",
            "actual_class": "Not available",
            "transaction": {
                feature_name: value - 0.5 for feature_name, value in dummy_base.items()
            },
        },
    ]

for sample in phase8_samples:
    prediction = predict_fraud(sample["transaction"])
    print(f"Sample type: {sample['sample_type']}")
    print(f"Actual class: {sample['actual_class']}")
    print(f"Fraud probability: {prediction['fraud_probability']:.6f}")
    print(f"Decision: {prediction['decision']}")
    print(f"Risk level: {prediction['risk_level']}")
    print(f"Reason: {prediction['reason']}")
    print("-" * 60)


Sample type: Actual normal transaction
Actual class: 0
Fraud probability: 0.000000
Decision: APPROVE
Risk level: LOW
Reason: Transaction probability is below review threshold.
------------------------------------------------------------
Sample type: Actual fraud transaction
Actual class: 1
Fraud probability: 0.913333
Decision: BLOCK
Risk level: HIGH
Reason: Transaction probability is above block threshold.
------------------------------------------------------------
Sample type: Manually created borderline transaction
Actual class: Not available
Fraud probability: 0.056667
Decision: APPROVE
Risk level: LOW
Reason: Transaction probability is below review threshold.
------------------------------------------------------------


# Batch Prediction Test

This section adds a small batch inference helper for production-style testing with multiple rows at once.

It uses the saved model and saved decision policy only, without retraining or recalculating evaluation metrics.


In [80]:
def predict_batch(transactions_df):
    if not isinstance(transactions_df, pd.DataFrame):
        raise ValueError("Batch prediction failed: input must be a pandas DataFrame.")

    if transactions_df.empty:
        raise ValueError("Batch prediction failed: input DataFrame cannot be empty.")

    sample_id_series = None
    if "sample_id" in transactions_df.columns:
        sample_id_series = transactions_df["sample_id"].copy()

    for row_index, row in transactions_df.iterrows():
        validate_transaction_input(row, final_feature_columns)

    model_input_df = transactions_df.loc[:, final_feature_columns].copy()
    model_input_df = model_input_df.reindex(columns=final_feature_columns)
    model_input_df = model_input_df.apply(pd.to_numeric, errors="raise")

    fraud_probabilities = final_model.predict_proba(model_input_df)[:, 1]

    prediction_rows = []
    for row_number, probability in enumerate(fraud_probabilities):
        decision_result = apply_decision_policy(float(probability), final_decision_policy)

        prediction_row = {
            "fraud_probability": float(probability),
            "decision": decision_result["decision"],
            "risk_level": decision_result["risk_level"],
            "reason": decision_result["reason"],
        }

        if sample_id_series is not None:
            prediction_row["sample_id"] = sample_id_series.iloc[row_number]
        else:
            prediction_row["original_index"] = transactions_df.index[row_number]

        prediction_rows.append(prediction_row)

    predictions_df = pd.DataFrame(prediction_rows)

    id_column = "sample_id" if sample_id_series is not None else "original_index"
    ordered_columns = [id_column, "fraud_probability", "decision", "risk_level", "reason"]
    return predictions_df[ordered_columns]


phase9_dataset_path = PROJECT_ROOT / "data" / "processed" / "final_features.csv"

if phase9_dataset_path.exists():
    phase9_source_df = pd.read_csv(phase9_dataset_path).head(6).copy()
    phase9_batch_df = phase9_source_df[final_feature_columns].copy()
    phase9_batch_df.insert(0, "sample_id", [f"sample_{i}" for i in range(len(phase9_batch_df))])
    phase9_actual_class = phase9_source_df["Class"].reset_index(drop=True)
else:
    phase9_dummy_rows = []
    for row_number in range(5):
        phase9_dummy_rows.append(
            {
                "sample_id": f"dummy_{row_number}",
                **{
                    feature_name: float(index + row_number)
                    for index, feature_name in enumerate(final_feature_columns, start=1)
                },
            }
        )

    phase9_batch_df = pd.DataFrame(phase9_dummy_rows)
    phase9_actual_class = pd.Series(["Not available"] * len(phase9_batch_df))

phase9_predictions = predict_batch(phase9_batch_df)
phase9_display_df = phase9_predictions.copy()
phase9_display_df.insert(1, "actual_class", phase9_actual_class.values)
phase9_display_df


,sample_id,actual_class,fraud_probability,decision,risk_level,reason
0,sample_0,0,0.0,APPROVE,LOW,Transaction probability is below review thresh...
1,sample_1,0,0.0,APPROVE,LOW,Transaction probability is below review thresh...
2,sample_2,0,0.0,APPROVE,LOW,Transaction probability is below review thresh...
3,sample_3,0,0.0,APPROVE,LOW,Transaction probability is below review thresh...
4,sample_4,0,0.0,APPROVE,LOW,Transaction probability is below review thresh...
5,sample_5,0,0.0,APPROVE,LOW,Transaction probability is below review thresh...
